<a href="https://colab.research.google.com/github/Rohan-1103/PySparkTutorial/blob/main/6_Example_Of_Pyspark_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Examples Of Pyspark ML

In [58]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Missing').getOrCreate()

In [59]:
## Read The dataset
training = spark.read.csv('test1.csv',header=True,inferSchema=True)

In [60]:
training.show()



+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [61]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [62]:
training.columns

['Name', 'age', 'Experience', 'Salary']

In [63]:
# [Age,Experience]----> new feature--->independent feature

In [64]:
from pyspark.ml.feature import VectorAssembler
featureassembler=VectorAssembler(inputCols=["age","Experience"],outputCol="Independent Features")

In [65]:
output=featureassembler.transform(training)

In [66]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent Features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [67]:
output.columns

['Name', 'age', 'Experience', 'Salary', 'Independent Features']

In [68]:
finalized_data=output.select("Independent Features","Salary")

In [69]:
finalized_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



In [70]:
from pyspark.ml.regression import LinearRegression
##train test split
train_data,test_data=finalized_data.randomSplit([0.70,0.30])
regressor=LinearRegression(featuresCol='Independent Features', labelCol='Salary')
regressor=regressor.fit(train_data)

In [71]:
### Coefficients
regressor.coefficients

DenseVector([47.619, 1285.7143])

In [72]:
### Intercepts
regressor.intercept

13619.047619047727

In [73]:
### Prediction
pred_results=regressor.evaluate(test_data)

In [74]:
pred_results.predictions.show()

+--------------------+------+------------------+
|Independent Features|Salary|        prediction|
+--------------------+------+------------------+
|          [23.0,2.0]| 18000|17285.714285714283|
|         [31.0,10.0]| 30000| 27952.38095238098|
+--------------------+------+------------------+



In [75]:
pred_results.meanAbsoluteError,pred_results.meanSquaredError

(1380.952380952369, 2351473.922902441)

### Scikit-learn Approach

In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Convert PySpark DataFrame to Pandas DataFrame
pandas_df = finalized_data.toPandas()

# Prepare features and target for Scikit-learn
X = pd.DataFrame(pandas_df['Independent Features'].tolist(), columns=['age', 'Experience'])
y = pandas_df['Salary']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print(f"Scikit-learn Train data shape: {X_train.shape}")
print(f"Scikit-learn Test data shape: {X_test.shape}")

# Initialize and train the Linear Regression model
sk_regressor = LinearRegression()
sk_regressor.fit(X_train, y_train)

print("\nScikit-learn Coefficients:", sk_regressor.coef_)
print("Scikit-learn Intercept:", sk_regressor.intercept_)


Scikit-learn Train data shape: (4, 2)
Scikit-learn Test data shape: (2, 2)

Scikit-learn Coefficients: [-714.28571429 3485.71428571]
Scikit-learn Intercept: 26857.142857142866


In [77]:
# Make predictions on the test set
y_pred = sk_regressor.predict(X_test)

# Evaluate the model
sk_mae = mean_absolute_error(y_test, y_pred)
sk_mse = mean_squared_error(y_test, y_pred)

print(f"\nScikit-learn Mean Absolute Error: {sk_mae}")
print(f"Scikit-learn Mean Squared Error: {sk_mse}")

# Display predictions (first 5)
predictions_df = pd.DataFrame({'Actual Salary': y_test, 'Predicted Salary': y_pred})
display(predictions_df.head())



Scikit-learn Mean Absolute Error: 8942.857142857156
Scikit-learn Mean Squared Error: 80369795.9183676


,Actual Salary,Predicted Salary
0,30000,39571.428571
1,25000,33314.285714


### PySpark ML vs. Scikit-learn: Career Advantages and Disadvantages

Here's a comparison of PySpark ML and Scikit-learn from a career perspective, helping you understand when to leverage each technology:

| Feature/Aspect      | PySpark ML                                                                           | Scikit-learn                                                                        |
| :------------------ | :----------------------------------------------------------------------------------- | :---------------------------------------------------------------------------------- |
| **Best Suited For** | Large-scale data, Big Data projects, Distributed computing, Real-time analytics, Cloud-based ML pipelines. | Small to medium datasets, Local machine development, Rapid prototyping, Traditional ML tasks, Academic research. |
| **Scalability**     | **Advantage:** Highly scalable, designed for distributed processing across clusters.   | **Disadvantage:** Limited to single-machine resources, not suitable for Big Data directly. |
| **Performance**     | **Advantage:** Can process vast amounts of data quickly due to parallelization.      | **Advantage:** Often faster for small to medium datasets due to optimized C/Python implementations. |
| **Learning Curve**  | **Disadvantage:** Steeper learning curve due to distributed computing concepts, Spark ecosystem complexities. | **Advantage:** Relatively easier to learn and use, intuitive API, extensive documentation. |
| **Job Market**      | **Advantage:** High demand for roles involving Big Data, Data Engineering, Distributed ML, MLOps in large enterprises. | **Advantage:** Fundamental for most Data Scientist and ML Engineer roles, widely applicable across industries. |
| **Ecosystem**       | Integrates well with Hadoop, Kafka, Delta Lake; strong in data engineering workflows.   | Rich ecosystem with tools like Pandas, NumPy, Matplotlib, Seaborn for data manipulation and visualization. |
| **Debugging**       | **Disadvantage:** More complex due to distributed nature, requires understanding Spark UI and logs. | **Advantage:** Easier to debug as it runs on a single machine.                       |
| **Resource Mgmt.**  | Requires cluster management skills (e.g., Yarn, Kubernetes), resource allocation.      | Minimal resource management beyond local machine capacity.                          |
| **Key Use Cases**   | Building large-scale recommendation systems, fraud detection, complex ETL processes on Big Data. | Developing predictive models for smaller datasets, exploratory data analysis, quick model iteration. |
